# Lab 2: Measurement & Indexes — Diagnostic Lab

**ECON 5200: Causal Machine Learning & Applied Analytics**

This lab uses a **diagnosis-first** format. You will:
1. Find and fix 4 bugs in a broken deflation pipeline
2. Compare CPI-U vs. Chained CPI (C-CPI-U) to quantify substitution bias
3. Measure Goodhart's Law on a two-metric KPI dashboard
4. Save your fixed function as a module (`deflation_utils.py`)

**Time estimate:** about 70 minutes (Part 0: 15 min, Parts 1–3: 15 min each, Part 4: 10 min)

**Prerequisites:** Chapter 2 reading, including [5200-DEPTH] sections on index number theory and KPI taxonomy

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

> ### What you write, and what you run
>
> **You write:**
> - **Part 1, the fix cell.** The body of `deflate_series_fixed`: about five lines. You also explain each of the four bugs in a sentence or two.
> - **Parts 2 and 3:** two one-line blanks in each (the two inflation rates, then the two correlations), and five interpretation answers in text cells.
> - **Part 4, the module.** Paste your Part 1 body into the template and add a two-line check, then paste in Lab 1's `profile_dataframe()` and give it type hints and a docstring.
> - **The AI Expansion:** the AI writes that code; you write the three answers about what it got wrong.
>
> **You run and read:** everything else. Part 0, the broken function, the downloads, the charts and the check cells are working code to learn from, not to retype.
>
> Writing code from a blank cell is what Problem Set 1 grades. Its ungraded warm-up, linked from the PS1 assignment, is the practice. If you are reading this lab after its due date, read it next to the Lab 2 solutions as a worked reference for PS1.

## Setup

`load_fred()` downloads one series from FRED's public CSV link, so this lab
needs no API key and no account.

**Optional: the `fredapi` package.** It does the same job with a free API key,
and adds search, metadata and vintages. Nothing in this lab needs it. If you
want it, the key goes in Colab's secret store, never in the notebook: left
sidebar → key icon (**Secrets**) → **Add new secret**, name `FRED_API_KEY`,
notebook access on. Get a free key at https://fredaccount.stlouisfed.org/apikeys.

```python
!pip install fredapi -q
from google.colab import userdata
from fredapi import Fred
fred = Fred(api_key=userdata.get("FRED_API_KEY"))
fred.get_series("CPIAUCSL", observation_start="1964-01-01")
```

Written this way the key is never in the notebook, so there is nothing to strip
before you push. Never type a key into a cell or paste one into an AI chat, and
never leave one on screen: a screen share or a recording captures it. If that
happens, revoke it and get a new one at the link above.

In [ ]:
# GIVEN — run as-is

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def load_fred(series_id, observation_start="1947-01-01"):
    """Return one FRED series as a pandas Series indexed by date."""
    url = ("https://fred.stlouisfed.org/graph/fredgraph.csv"
           f"?id={series_id}&cosd={observation_start}")
    frame = pd.read_csv(url)
    # name the columns by position: FRED has renamed the value column before
    frame.columns = ["date", series_id]
    frame["date"] = pd.to_datetime(frame["date"])
    # errors="coerce" turns anything that is not a number into NaN;
    # FRED leaves a month it has no value for blank, and dropna() removes it
    frame[series_id] = pd.to_numeric(frame[series_id], errors="coerce")
    return frame.set_index("date")[series_id].dropna()


print("Setup complete (no API key needed)")

<!-- 5200-onramp -->
## Part 0: Build It First (GUIDED — 15 min)

Part 4 asks you to save your fixed function in `deflation_utils.py`, a
**module**: a file of Python that other code can `import`. This part builds up
to it with functions — `def`, parameters, default arguments, docstrings,
`return`, and raising an error on purpose — and the two ways a notebook turns a
cell into an importable `.py` file.

*(If you have not done Lab 1's Part 0, do it first: it covers f-strings, the
DataFrame, and boolean masks, all of which are assumed here.)*

### Step 0a: writing a function

A **function** is a named block of code you can reuse. `def` starts it, the
indented lines are its body, and `return` hands a value back to whoever called
it.

- The triple-quoted string at the top of the body is a **docstring**:
  documentation, not a comment. `help()` prints it, and every function in the
  modules you write this term needs one.
- A **default argument** (`rate=0.02`) makes a parameter optional. Pass a new
  value by name (`rate=0.05`) or by position to override it.
- A function with no `return` hands back `None`. If you see
  `AttributeError: 'NoneType' object has no attribute ...`, you almost always
  used the result of a function that forgot to return anything.

In an f-string, `{x:.2f}` prints `x` with two decimals, and `{x:.1%}` prints a
fraction as a percent with one decimal.

In [ ]:
# GIVEN — run as-is
# Step 0a: writing a function

def real_rate(nominal, inflation_pct):
    """Return the real interest rate by the Fisher approximation.

    Both arguments are in percent: 5.0 means 5%.
    """
    return nominal - inflation_pct

print(f"US:     {real_rate(5.25, 3.10):.2f}%")
print(f"Brazil: {real_rate(10.75, 4.50):.2f}%")

help(real_rate)


def compound(start, rate=0.02, years=5):
    """Grow `start` at `rate` a year for `years` years."""
    return start * (1 + rate) ** years

print(f"\nDefaults:            {compound(100):.2f}")
print(f"Override the rate:   {compound(100, rate=0.05):.2f}")
print(f"Override both:       {compound(100, 0.05, 10):.2f}")


def just_prints(x):
    print(f"  (printed {x}, returned nothing)")

print("\nWhat did just_prints give back?", just_prints(3))

### Step 0b: a correct `deflate()`, and a deliberate error

This is the function Part 1 shows you with bugs in it, and the one you save as
a module in Part 4. Build the correct version first:

    Real value = (Nominal / CPI) × CPI in the base year

Both halves must be in the **same units** and aligned on the **same dates**.
The classic errors are mixing an annual average with a monthly reading, and
dividing by the base CPI instead of multiplying by it.

`nominal` and `cpi` are **parameters**: names that exist only inside the
function. That is why they can be short here, while the series outside are
called `demo_nominal` and `demo_cpi`.

When the base year is missing, `deflate()` **raises** an error instead of
returning all-`NaN`, because a silent `NaN` would travel downstream into a
published number. `raise ValueError(...)` stops with that message;
`try` / `except ValueError` catches it so the notebook keeps going. The last
lines of the cell show the guard at work.

The demo CPI starts at 100 and rises 0.25% a month, so this part needs no
download.

In [ ]:
# GIVEN — run as-is
# Step 0b: a correct deflate(), and a deliberate error

demo_dates = pd.date_range("2018-01-01", periods=84, freq="MS")   # 84 month starts
demo_cpi = pd.Series(100 * 1.0025 ** np.arange(84), index=demo_dates)
demo_nominal = pd.Series(20 + 0.05 * np.arange(84), index=demo_dates)


def deflate(nominal, cpi, base_year=2020):
    """Convert a nominal series to constant base_year dollars."""
    base = cpi.loc[cpi.index.year == base_year]
    if len(base) == 0:
        raise ValueError(f"base_year {base_year} not present in the CPI index")
    return (nominal / cpi) * base.mean()


demo_real = deflate(demo_nominal, demo_cpi, base_year=2020)
base_2020 = demo_cpi[demo_cpi.index.year == 2020].mean()

print(f"base CPI (2020 mean): {base_2020:.2f}")
print(f"first {demo_dates[0].date()}: nominal ${demo_nominal.iloc[0]:.2f} -> real ${demo_real.iloc[0]:.2f}")
print(f"last  {demo_dates[-1].date()}: nominal ${demo_nominal.iloc[-1]:.2f} -> real ${demo_real.iloc[-1]:.2f}")
print(f"\nnominal growth: {demo_nominal.iloc[-1] / demo_nominal.iloc[0] - 1:.1%}")
print(f"real growth:    {demo_real.iloc[-1] / demo_real.iloc[0] - 1:.1%}")

try:
    deflate(demo_nominal, demo_cpi, base_year=1970)
except ValueError as e:
    print(f"\ndeflate(..., base_year=1970) -> ValueError: {e}")

### How a notebook makes a real file

Part 4 asks for `deflation_utils.py`. There are two ways to write it from a
notebook. This course uses the first; you will meet the second in other
people's code.

**Route 1 — the `%%writefile` magic.** What Part 4, Labs 3–5 and the Problem
Set 1 starter all do. A cell whose *first line* is `%%writefile
deflation_utils.py` writes the rest of that cell to disk instead of running it:

```python
%%writefile deflation_utils.py
"""Deflation utilities."""
import pandas as pd

def deflate_series(nominal, cpi, base_year=2020):
    ...
```

**Route 2 — a string, then write it.** The module goes inside a
triple-quoted string, and `with open("deflation_utils.py", "w") as f:` writes
the string to a file. The catch: it is an ordinary string, so a `"\n"` inside
your code turns into a real line break, and the error that follows points at a
line you never wrote. `%%writefile` has no such problem.

Either way the next cell can `import deflation_utils` like any other library.
Two consequences worth knowing before you hit them:

- The file gets **only what is in that cell**. Imports the functions need must
  be written inside it: the notebook's earlier `import pandas as pd` does not
  travel into the file.
- Python caches imported modules. Writing the file again after an edit does not
  change the version already imported; you need `importlib.reload(module)`, or
  *Restart session*. That is why Part 4 calls `importlib.reload` right after
  the import.

### Now diagnose

You have a correct `deflate()` and its output on screen. Part 1's version
computes something subtly different. Run it, compare the *numbers* against
what you just produced, and only then go looking for the line.

---

## Part 1: Find and Fix the Bugs (DIAGNOSTIC — 15 min)

The following code attempts to deflate nominal wages using CPI data from FRED. It contains **4 bugs** — common errors that even experienced analysts make. Your job: find each bug, explain why it's wrong, and fix it.

**Hint:** The bugs are related to:
- Base period selection
- Date alignment between the two series
- Missing data handling
- The units the output is actually in

Compare the output against the `deflate()` you built in Step 0b before you
read the function line by line.

In [ ]:
# BROKEN CODE — find and fix the 4 bugs

def deflate_series_broken(nominal, cpi, base_year=2020):
    """Convert a nominal series to constant base_year dollars."""
    base_cpi = cpi.loc[f"{base_year}-01-01"]

    real = (nominal / cpi) * base_cpi

    real = nominal / cpi * 100

    return real

In [ ]:
# RUN THE BROKEN FUNCTION — look at what it reports

wages = load_fred("AHETPI", observation_start="1964-01-01")       # hourly wage, production workers
cpi_data = load_fred("CPIAUCSL", observation_start="1964-01-01")  # CPI-U, seasonally adjusted

result = deflate_series_broken(wages, cpi_data, base_year=2020)

# .isna() is True for each missing value, and .sum() counts the Trues
print(f"Result range: {result.min():.2f} to {result.max():.2f}")
print(f"Rows returned: {len(result)}   of which NaN: {result.isna().sum()}")
print(f"Wage months: {len(wages)}   CPI months: {len(cpi_data)}")

### Your Fixes

For each bug, explain:
1. **What's wrong** (1 sentence)
2. **Why it matters** (what does the error do to the output?)
3. **The fix** (show the corrected code)

*Your answers here:*

**Bug 1 —**

**Bug 2 —**

**Bug 3 —**

**Bug 4 —**

In [ ]:
# YOUR FIX — write the corrected function

def deflate_series_fixed(nominal, cpi, base_year=2020):
    """Convert a nominal series to constant base_year dollars.

    Returns a Series with no missing values, on the dates where both inputs
    are observed.
    """
    # YOUR CODE HERE — one fix for every bug you documented above
    pass


wages_real = deflate_series_fixed(wages, cpi_data, base_year=2020)

# {date:%b %Y} prints a date as "Aug 2026"
print(f"Real wage range: ${wages_real.min():.2f} to ${wages_real.max():.2f} (2020 $)")
print(f"Real wage in Jan 1973: ${wages_real.loc['1973-01-01']:.2f}")
print(f"Real wage latest ({wages_real.index[-1]:%b %Y}): ${wages_real.iloc[-1]:.2f}")

### Verification Checkpoint

Measured against live FRED (FRED revises, so the second decimal can move):

| | 2020 dollars |
|---|---|
| Jan 1973 | **\$24.43** |
| Latest month | about **\$25.20** |
| Series minimum | \$19.71 (1995) |
| Series maximum | **\$25.42, April 2020** — a composition effect, not a raise |

Before 2000 the peak is January 1973; the all-time maximum is April 2020.

Then run the check below. It tests the arithmetic rather than a hard-coded
number, so it keeps working as FRED revises. It works out which base CPI your
result implies on every date, and compares that with the one `base_year` calls
for. The second call repeats the test for 2017.

`assert condition, message` stops with `message` when the condition is False.
`np.allclose` compares two sets of numbers up to rounding error.

In [ ]:
# CHECK — run after your fix

def check_deflated(real, nominal, cpi, base_year):
    """Raise AssertionError unless `real` is `nominal` in base_year dollars."""
    assert real is not None, "Got None: the function has no return statement yet."
    n_nan = int(real.isna().sum())
    assert n_nan == 0, f"{n_nan} NaN row(s) in the result."
    rows = real.index.intersection(nominal.index).intersection(cpi.index)   # dates in all three
    assert len(rows) == len(real), (
        f"{len(real) - len(rows)} row(s) fall on dates missing from the wage or CPI series.")
    implied_base = real[rows] / nominal[rows] * cpi[rows]
    target = cpi[cpi.index.year == base_year].mean()
    assert np.allclose(implied_base, target, rtol=1e-6), (
        f"The result is scaled to a CPI of {implied_base.median():.2f}; "
        f"base_year={base_year} requires {target:.2f}.")


check_deflated(wages_real, wages, cpi_data, base_year=2020)
check_deflated(deflate_series_fixed(wages, cpi_data, base_year=2017),
               wages, cpi_data, base_year=2017)
print("Part 1 checks passed: no NaN, aligned dates, and the base_year argument is honoured.")

## Part 2: CPI-U vs. Chained CPI — Quantifying Substitution Bias (EXTEND — 15 min)

The chapter discussed how the Laspeyres CPI overstates inflation because of
substitution bias. The BLS publishes the Chained CPI (C-CPI-U, FRED series
`SUUR0000SA0`) as a partial correction. This part measures the gap.

**Compare like with like.** C-CPI-U is published **not seasonally adjusted**,
with **December 1999 = 100**. The CPI-U you used in Part 1 (`CPIAUCSL`) is
seasonally adjusted, so this part pulls the unadjusted CPI-U, `CPIAUCNS`, from
the same month. Mixing the two makes the end-point gap swing by more than a
full index point depending on which month the data happens to end in.

The last 10–12 months of C-CPI-U are **preliminary** and are revised as
expenditure data arrive, so the last decimal of any gap you compute is
provisional.

In [ ]:
# GIVEN — run as-is: both CPI variants, not seasonally adjusted, from Dec 1999

cpi_u = load_fred("CPIAUCNS", observation_start="1999-12-01")      # CPI-U
cpi_c = load_fred("SUUR0000SA0", observation_start="1999-12-01")   # C-CPI-U, the chained CPI

print(f"CPI-U observations: {len(cpi_u)}")
print(f"C-CPI-U observations: {len(cpi_c)}")

In [ ]:
# GIVEN — run as-is: rebase both indexes to 100 in December 1999, and plot

# A dict of Series becomes a DataFrame with one column per key, lined up on
# dates; dropna() keeps only the months both indexes have.
combined = pd.DataFrame({"CPI-U": cpi_u, "C-CPI-U": cpi_c}).dropna()
combined_norm = combined / combined.iloc[0] * 100      # divide every row by the first

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(combined_norm.index, combined_norm["CPI-U"], label="CPI-U (standard, NSA)", color="red")
ax.plot(combined_norm.index, combined_norm["C-CPI-U"], label="C-CPI-U (chained)", color="blue")
ax.set_xlabel("Year")
ax.set_ylabel("Index (Dec 1999 = 100)")
ax.set_title("CPI-U vs. Chained CPI: Measuring Substitution Bias")
ax.legend()
plt.show()

In [ ]:
# GIVEN — run as-is: the gap in index points

# subtracting two dates gives a time span; .days / 365.25 turns it into years
years = (combined_norm.index[-1] - combined_norm.index[0]).days / 365.25
gap_points = combined_norm["CPI-U"].iloc[-1] - combined_norm["C-CPI-U"].iloc[-1]

print(f"Cumulative gap over {years:.1f} years: {gap_points:.1f} index points")
print(f"That is {gap_points / years:.2f} index points per year.")

### Put the gap in the right units

The Boskin Commission, the BLS and the chapter all state substitution bias in
**percentage points of inflation per year**. Index points per year is not that
quantity. Fill in the two blanks below: the average annual inflation rate of
each index over `years`, in percent. The cell takes the difference.

A pointer: your blank undoes Step 0a's `compound()`. Which `rate` takes 100 to the index's end value in `years`?

In [ ]:
# YOUR TURN — two blanks: each index's average annual inflation rate

cpi_u_rate = ____      # CPI-U, percent per year
cpi_c_rate = ____      # C-CPI-U, percent per year
annual_gap_pp = cpi_u_rate - cpi_c_rate

print(f"CPI-U:   {cpi_u_rate:.2f}% per year")
print(f"C-CPI-U: {cpi_c_rate:.2f}% per year")
print(f"Upper-level substitution bias: {annual_gap_pp:.2f} pp per year")

In [ ]:
# CHECK — run after the two blanks

for label, rate in [("CPI-U", cpi_u_rate), ("C-CPI-U", cpi_c_rate)]:
    end = combined_norm[label].iloc[-1]
    # Accept either convention: annual compounding or a continuous (log) rate.
    annual = 100 * (1 + rate / 100) ** years
    continuous = 100 * np.exp(rate / 100 * years)
    assert min(abs(annual - end), abs(continuous - end)) < 0.5, (
        f"{label}: {rate:.3f}% a year for {years:.1f} years does not take the "
        f"index from 100 to {end:.1f}. Check the units: percent, per year.")
print(f"Part 2 check passed: {annual_gap_pp:.2f} pp per year.")

### Interpretation Question

**Q1:** The Boskin Commission (1996) put total substitution bias at about
0.4 pp per year: **0.15 pp upper-level** (substitution *between* item
categories) and **0.25 pp lower-level** (substitution *within* a category).
Since January 1999 the CPI-U itself uses a geometric mean for most lower-level
indexes. Which of Boskin's two components does your CPI-U vs C-CPI-U gap
measure, and is your estimate above or below Boskin's figure for it? Give one
reason the 2000–present number might differ from a 1996 estimate.

**Q2:** Your print-out above shows both index points per year and percentage
points per year. Explain in two sentences why they differ, and which one a
cost-of-living adjustment compounds.

**Q3:** Social Security COLAs are indexed to CPI-W, a close relative of CPI-U.
Treat the two as equal, and use your pp-per-year estimate: by roughly what
percentage would a benefit be lower after a 20-year retirement if COLAs
followed C-CPI-U instead? Federal tax brackets have been indexed to C-CPI-U
since 2017 — who gains and who loses from that switch?

*Your answers here:*



## Part 3: KPI Dashboard — Detecting Goodhart's Law (BUILD — 15 min)

A dashboard that watches a primary metric **and** a counter-metric can catch
gaming when it starts. The data here are simulated: DAU/MAU (daily active users
over monthly active users, an engagement ratio) and time per session.

For the first 18 months both metrics grow organically. From month 18 the team
pushes notifications: DAU/MAU climbs faster while sessions get shorter.

In the code, `np.random.normal(0, 0.01, n)` draws `n` noise terms with standard
deviation 0.01, and `np.concatenate` joins the two phases end to end. Fixing
the seed makes the "random" noise the same on every run.

In [ ]:
# GIVEN — run as-is: simulate DAU/MAU rising while session time falls

np.random.seed(42)
phase_len = 18                            # months in each phase
dates = pd.date_range("2022-01-01", periods=2 * phase_len, freq="MS")
t = np.arange(phase_len)                  # months into the phase: 0, 1, ..., 17

dau_mau = np.concatenate([
    0.45 + 0.005 * t + np.random.normal(0, 0.01, phase_len),   # organic
    0.54 + 0.008 * t + np.random.normal(0, 0.01, phase_len),   # notification push
])
time_per_session = np.concatenate([
    8.0 + 0.15 * t + np.random.normal(0, 0.3, phase_len),      # organic
    10.7 - 0.20 * t + np.random.normal(0, 0.3, phase_len),     # sessions shorten
])

kpi_data = pd.DataFrame({
    "DAU/MAU": dau_mau,
    "Time per Session (min)": time_per_session,
}, index=dates)

print(kpi_data.describe().round(2))

In [ ]:
# GIVEN — run as-is: the dashboard, with a dashed line where gaming begins

def plot_metric(ax, data, column, title, ylabel, colour, break_date):
    """Plot one column of `data` on `ax`, with a dashed line at `break_date`."""
    ax.plot(data.index, data[column], color=colour, marker="o", markersize=3)
    ax.axvline(break_date, color="red", linestyle="--", label="Gaming begins")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend()


gaming_start = dates[phase_len]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))     # two panels side by side
plot_metric(ax1, kpi_data, "DAU/MAU", "Primary metric: DAU/MAU",
            "DAU/MAU ratio", "tab:blue", gaming_start)
plot_metric(ax2, kpi_data, "Time per Session (min)", "Counter-metric: time per session",
            "Minutes", "tab:orange", gaming_start)
fig.suptitle("Detecting Goodhart's Law: When Metrics Diverge")
plt.tight_layout()
plt.show()

### Measure the divergence

The chart shows the two metrics splitting apart; a monitoring system needs a
number. Fill in the correlation between DAU/MAU and time per session within
each phase. The first `phase_len` rows are the organic phase and the rest
are the gaming phase. `.iloc[a:b]` slices rows by position, like a list
slice. Look up `.corr()`.

In [ ]:
# YOUR TURN — two blanks: the correlation in each phase

corr_organic = ____
corr_gaming = ____

# :+.2f prints the sign even when the number is positive
print(f"Correlation (organic phase): {corr_organic:+.2f}")
print(f"Correlation (gaming phase):  {corr_gaming:+.2f}")

### Interpretation Question

**Q1:** In the organic phase, DAU/MAU and time-per-session are positively correlated. In the gaming phase, they become negatively correlated. Explain why this sign flip is a diagnostic signal for Goodhart's Law.

**Q2:** You are a product economist at this company. The VP of Growth shows the DAU/MAU chart (left panel only) and says, "Engagement is up 30% — great quarter!" What would you say?

*Your answers here:*



## Part 4: Write a Verification Module (PRODUCTION — 10 min)

Save your fixed function as `deflation_utils.py`, a module you can import in
later labs. The next cell is a `%%writefile` cell for you to finish. Two things
go in it:

1. **`deflate_series()`.** Copy the body of your Part 1 `deflate_series_fixed`
   into the template, under the docstring and the base-year guard that are
   already there. The docstring also promises a `ValueError` when the two
   series share no dates, so add that check once you have lined them up.
2. **`profile_dataframe()`** from Lab 1 Part 3. Paste it in underneath, and give
   it type hints and a docstring like `deflate_series()` has. It is the function
   you will reach for first on every new dataset this term.

The template labels its arguments with **type hints**: `nominal: pd.Series`
means "this should be a pandas Series", and `-> pd.Series` says what comes
back. Python does not enforce them; they are there for the reader.

The check cell after it reuses Part 1's `check_deflated`, so the module is
tested against the definition of a base-year deflation, not against your own
earlier output.

In [ ]:
%%writefile deflation_utils.py
"""Deflation utilities for ECON 5200 labs."""

import pandas as pd


def deflate_series(nominal: pd.Series, cpi: pd.Series,
                   base_year: int = 2020) -> pd.Series:
    """Convert a nominal series to constant base_year dollars.

    Raises ValueError if base_year is missing from `cpi` or the series share no dates.
    """
    # .any() is True if at least one CPI date falls in base_year
    if not (cpi.index.year == base_year).any():
        raise ValueError(f"Base year {base_year} not found in CPI data.")

    # YOUR CODE HERE — the body of your deflate_series_fixed from Part 1
    pass


# YOUR CODE HERE — paste profile_dataframe() from Lab 1 Part 3

In [ ]:
# CHECK — import the module and test it

import importlib
import deflation_utils
importlib.reload(deflation_utils)      # picks up your edits when you re-run the %%writefile cell

check_deflated(deflation_utils.deflate_series(wages, cpi_data, base_year=2020),
               wages, cpi_data, base_year=2020)
print("Module passes the Part 1 check.")

---
## AI-Assisted Expansion: Index-Integrity Monitor

**AI policy: foundations first, expansion second.** You have fixed a deflation
function, measured substitution bias in the right units and detected a Goodhart
sign flip by hand. From here on, AI may write code for you (the Co-Pilot Rule).

### Your Expansion Task
Build ONE interactive monitor, inside this notebook (ipywidgets + plotly), that lets the user:
1. Pick a start date and see the CPI-U vs C-CPI-U gap recomputed in **both** index points per year and percentage points per year
2. Switch between the seasonally adjusted and unadjusted CPI-U and see why the gap moves
3. Set a rolling-window length and watch the rolling correlation between DAU/MAU and time per session
4. Get an alert on the first window where that correlation turns negative

Use the functions you already built; do not write a second module.

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [ ]:
# AI EXPANSION — copy the prompt below into Claude, then paste
# its code at the bottom of this cell. Run it and check it.

# [Prep] Act as an expert Python data scientist specializing
# in price-index measurement and KPI monitoring.
#
# [Request] I just completed a diagnosis-first lab where I fixed
# a four-bug deflation function (deflate_series_fixed), measured
# the CPI-U vs C-CPI-U substitution bias as a difference of
# annual inflation rates in percentage points per year, and
# detected a Goodhart sign flip between DAU/MAU and time per
# session. Build ONE interactive monitor that runs in Colab:
#   1. a start-date picker that recomputes the CPI-U vs C-CPI-U
#      gap as index points per year AND as pp per year
#   2. a toggle between CPIAUCNS and CPIAUCSL for CPI-U
#   3. a rolling-window slider for the correlation between the
#      two columns of kpi_data
#   4. an alert naming the first window where it turns negative
#
# [Iterate] Use ipywidgets, plotly and pandas. Reuse my names:
# load_fred, cpi_u, cpi_c, combined_norm, kpi_data,
# deflation_utils.deflate_series. No Streamlit, no new .py file.
#
# [Mechanism Check] Add inline comments explaining:
#   - why index points per year is not a rate, and how the pp
#     figure is computed from compounding
#   - why C-CPI-U must be compared with the unadjusted CPI-U
#   - how the rolling window trades detection speed for noise
#
# [Evaluate] Explain which Boskin component the gap measures,
# and what a monitoring team should do when the alert fires.

# PASTE AI-GENERATED CODE BELOW:

### Document the interaction

Three short answers. This is what is graded for the expansion, not the code.

1. **What did the AI get wrong or leave out?** Name one line you had to change,
   or one claim in its explanation that your own output contradicts.
2. **How did you verify it?** Point to a number the dashboard shows that you
   also computed by hand earlier in this lab, and say whether they match.
3. **What would you ask differently next time?**

*Your answers here:*

1.
2.
3.

---
## Digital Portfolio: your README

### Generate your README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

In [ ]:
# README prompt — documentation only, no code

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Diagnosed and fixed four bugs in a deflation pipeline, including
#   output in 1982-84 dollars mislabelled as 2020 dollars
# * Measured upper-level substitution bias: CPI-U [YOUR VALUE]% a year vs
#   C-CPI-U [YOUR VALUE]% a year = [YOUR VALUE] pp per year, and explained
#   why the [YOUR VALUE] index-point gap per year is not that number
# * Detected Goodhart's Law as a correlation flip from [YOUR VALUE] to
#   [YOUR VALUE] between DAU/MAU and time per session
# * Shipped deflation_utils.py with a tested deflate_series()
# * Built an interactive index-integrity monitor
#
# **Please write a README.md entry including:**
# 1. Project Title: Index Integrity — Deflation, Substitution Bias & Goodhart
# 2. Objective: one sentence
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Write it plainly, in the first person. No words like 'robust',
# 'comprehensive' or 'leverage'. Use only the numbers I gave you."

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab02-deflation`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab02-deflation`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab02-deflation`, branch `main`,
   commit message `Lab 02 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. `deflation_utils.py` is **not** pushed by step 2 — that step sends the
   notebook only, and the file lives in Colab's temporary storage, which is
   wiped when the session ends. Download it first: in the Colab **Files** panel
   (folder icon, left), right-click `deflation_utils.py` → **Download**. Then on
   the repo page use **Add file → Upload files**, drag it in, and **Commit
   changes**. It is part of what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.

---

**End of Lab 2.** You fixed a deflation function that ran without an error and
still gave the wrong answer, measured substitution bias in percentage points
rather than index points, and watched two metrics that used to move together
split apart once one of them became the target.